In [21]:
import os
import ast
import json
import random
import certifi
import pandas as pd
from datasets import load_dataset

# Resiliência de I/O: Contorno de políticas estritas de SSL e desativação 
# de motores experimentais do HuggingFace (Xet) para prevenir falhas de download 
# em servidores de processamento sem terminal gráfico.
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Iniciando Fase 1 (Consolidada): Curadoria, Mapeamento Esparso e Hard Negatives")
print("=" * 70)


# =============================================================================
# BLOCO 1 — PARÂMETROS GLOBAIS
# =============================================================================

# --- Qualidade do EQ ---
LIMIAR_CONSISTENCIA_EQ = 0.500
# JUSTIFICATIVA CIENTÍFICA: A análise topológica do SocialFX revela que descritores 
# de timbre espectral (bright, dark) ultrapassam assintoticamente 0.80. A adoção 
# de um limiar mais permissivo incorpora termos com alta precisão descritiva localizados 
# na cauda longa da distribuição, maximizando a variância lexical de treino. A 
# desambiguação subsequente é garantida pela ortogonalidade introduzida nos Hard Negatives.

# --- Qualidade do Reverb ---
MIN_PALAVRAS_AGREED_REVERB = 1
# JUSTIFICATIVA CIENTÍFICA: A força de consenso para reverberação baseia-se na contagem 
# absoluta de palavras válidas retidas pelo crowdsourcing. Exigir >= 1 palavra válida 
# no atributo `agreed` expurga respostas sentinelas e amostras de descarte oriundas dos Turkers.

MAX_VIRGULAS_TEXT_REVERB = 2
# JUSTIFICATIVA CIENTÍFICA: Modelos Joint Embeddings (LAION-CLAP) empregam codificadores 
# de texto subword-based (RoBERTa). Rótulos multi-palavras densamente concatenados degradam 
# os mapas de atenção bidirecional, induzindo estados OOV latentes. O limite rígido de 
# 2 vírgulas consolida compostos sintaticamente processáveis.

TOP_K_POR_SPLIT = 300
# Estabelece um teto de expansão do vocabulário, estabilizando o uso da memória de 
# projeção textual no pipeline de treinamento contrastivo.

# --- Escala do NSynth ---
NOTAS_POR_FAMILIA = 60
# JUSTIFICATIVA CIENTÍFICA: Extração de amostras controladas por família 
# instrumental evita viés associativo condicional no preditor acústico.

PARES_POR_NOTA = 2
# A geração de 2 pares por nota dimensiona o lote total, garantindo volume 
# estatístico adequado após a propagação de 5 intensidades na etapa DSP.


# =============================================================================
# BLOCO 2 — MAPEAMENTO ESPARSO POR GRUPO ACÚSTICO
# =============================================================================

# JUSTIFICATIVA CIENTÍFICA: Alinhamento ontológico. Modelos perceptuais 
# respondem diferencialmente ao timbre espectral vs. temporal. O mapeamento restringe 
# os descritores alveiares a componentes cujos envelopes de ataque e decaimento 
# favoreçam a perturbação de sinal.
MAPEAMENTO_ESPARSO = {
    "cordas_e_guitarras": {
        "nsynth_families": ["guitar", "bass", "string", "synth_lead"],
        "adjetivos_alvo": ["bright", "dark", "warm", "tinny", "muffled", "crisp", "clear"]
    },
    "teclas_e_espacialidade": {
        "nsynth_families": ["keyboard", "mallet", "organ"],
        "adjetivos_alvo": ["spacious", "dry", "echo", "distant", "hollow", "warm", "clear"]
    },
    "sopros_e_textura": {
        "nsynth_families": ["brass", "flute", "reed", "vocal"],
        "adjetivos_alvo": ["harsh", "soft", "metallic", "mellow", "warm", "dark", "bright"]
    }
}

# JUSTIFICATIVA CIENTÍFICA: Mapeamento rígido de antonímia semântica. 
# A detecção de anomalias no espaço latente é corrigida providenciando o oposto 
# explícito na matriz de entrada. Apenas descritores atômicos (1 palavra) são aptos.
PARES_OPOSTOS = {
    "bright":   "dark",     "dark":     "bright",
    "warm":     "tinny",    "tinny":    "warm",
    "muffled":  "crisp",    "crisp":    "muffled",
    "spacious": "dry",      "dry":      "spacious",
    "echo":     "dry",
    "distant":  "clear",    "clear":    "distant",
    "harsh":    "soft",     "soft":     "harsh",
    "metallic": "mellow",   "mellow":   "metallic",
    "hollow":   "full",     "full":     "hollow",
}


# =============================================================================
# BLOCO 3 — FUNÇÕES AUXILIARES
# =============================================================================

def parse_extra_data(extra_str):
    """Extração sintaticamente segura de dicionários aninhados em formato string."""
    if isinstance(extra_str, str):
        try: return ast.literal_eval(extra_str)
        except (ValueError, SyntaxError): return {}
    return extra_str if isinstance(extra_str, dict) else {}

def parse_params(param_val):
    """Conversão de metadados paramétricos de áudio contidos em strings iteráveis."""
    if isinstance(param_val, str):
        try: return ast.literal_eval(param_val)
        except (ValueError, SyntaxError): return []
    return param_val if param_val is not None else []

def calc_consensus_strength_reverb(extra_dict):
    """Avalia a densidade de respostas expurgando artefatos de ausência de escolha."""
    agreed_raw = extra_dict.get('agreed', '')
    if not agreed_raw: return 0
    tokens_validos = [
        w.strip() for w in agreed_raw.split(',')
        if w.strip() and w.strip().lower() not in ('none', 'none of the above', '')
    ]
    return len(tokens_validos)


# =============================================================================
# BLOCO 4 — EXTRAÇÃO E FILTRAGEM DO SocialFX
# =============================================================================

print("\nBaixando SocialFX do Hugging Face...")
dataset_eq     = load_dataset("seungheondoh/socialfx-original", split="eq").to_pandas()
dataset_reverb = load_dataset("seungheondoh/socialfx-original", split="reverb").to_pandas()

# --- EQ ---
dataset_eq['extra_dict'] = dataset_eq['extra'].apply(parse_extra_data)
dataset_eq['lang']       = dataset_eq['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_eq['consistency'] = dataset_eq['extra_dict'].apply(lambda x: x.get('ratings_consistency', 0))

# JUSTIFICATIVA CIENTÍFICA: A blacklist visa estritamente artefatos não-acústicos 
# inseridos no crowdsourcing, limitando viés analítico ao preservar afecções válidas.
BLACKLIST_EQ = ['taco', 'baseball', 'disgusting', 'genius', 'poor']

df_eq_filtrado = dataset_eq[
    (dataset_eq['lang'] == 'English') &
    (dataset_eq['consistency'] >= LIMIAR_CONSISTENCIA_EQ) &
    (~dataset_eq['text'].isin(BLACKLIST_EQ))
].copy()
df_eq_filtrado = df_eq_filtrado.sort_values('consistency', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_eq_filtrado['param_values_list'] = df_eq_filtrado['param_values'].apply(parse_params)
df_eq_filtrado['tipo_efeito']       = 'eq'
df_eq_filtrado['consistencia']      = df_eq_filtrado['consistency']
df_eq_ouro = df_eq_filtrado.head(TOP_K_POR_SPLIT).copy()

print(f"  EQ     -> {len(df_eq_ouro):>4} termos | range consistência: [{df_eq_ouro.iloc[-1]['consistency']:.3f} - {df_eq_ouro.iloc[0]['consistency']:.3f}]")

# --- REVERB ---
dataset_reverb['extra_dict']         = dataset_reverb['extra'].apply(parse_extra_data)
dataset_reverb['lang']               = dataset_reverb['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_reverb['consensus_strength'] = dataset_reverb['extra_dict'].apply(calc_consensus_strength_reverb)

df_reverb_filtrado = dataset_reverb[
    (dataset_reverb['lang'] == 'English') &
    (dataset_reverb['consensus_strength'] >= MIN_PALAVRAS_AGREED_REVERB) &
    (dataset_reverb['text'].str.count(',') <= MAX_VIRGULAS_TEXT_REVERB)
].copy()

# JUSTIFICATIVA CIENTÍFICA: Transposição de tokens aglutinados para 
# separação ortográfica apropriada. A inserção de espaços assegura a interpretação de 
# múltiplos descritores como entidades semânticas compostas no Text Encoder.
df_reverb_filtrado['text'] = df_reverb_filtrado['text'].str.replace(',', ', ', regex=False)
df_reverb_filtrado = df_reverb_filtrado.sort_values('consensus_strength', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_reverb_filtrado['param_values_list'] = df_reverb_filtrado['param_values'].apply(parse_params)
df_reverb_filtrado['tipo_efeito']       = 'reverb'
df_reverb_filtrado['consistencia']      = df_reverb_filtrado['consensus_strength'].astype(float)
df_reverb_ouro = df_reverb_filtrado.head(TOP_K_POR_SPLIT).copy()

print(f"  Reverb -> {len(df_reverb_ouro):>4} termos | range consensus: [{int(df_reverb_ouro.iloc[-1]['consensus_strength'])} - {int(df_reverb_ouro.iloc[0]['consensus_strength'])} palavras]")

# Verificação de cobertura
todos_descritores_alvo = set(w for g in MAPEAMENTO_ESPARSO.values() for w in g['adjetivos_alvo'])
vocab_eq_set     = set(df_eq_ouro['text'].tolist())
vocab_reverb_set = set(df_reverb_ouro['text'].tolist())

print("\n  Cobertura dos descritores-alvo:")
for d in sorted(todos_descritores_alvo):
    if d in vocab_eq_set and d not in {"distant", "dry", "spacious", "echo"}:
        row = df_eq_ouro[df_eq_ouro['text'] == d].iloc[0]
        print(f"    ✓ {d:12s}  [EQ     | consistência={row['consistency']:.3f}]")
    elif d in vocab_reverb_set:
        row = df_reverb_ouro[df_reverb_ouro['text'] == d].iloc[0]
        print(f"    ✓ {d:12s}  [REVERB | consensus={int(row['consensus_strength'])}]")
    else:
        print(f"    ✗ {d:12s}  — ausente (verificar limiar ou split)")


# =============================================================================
# BLOCO 5 — VOCABULÁRIO DSP UNIFICADO
# =============================================================================

vocabulario_dsp = {}

# Trava semântica: Impede que o EQ capture palavras que devem obrigatoriamente
TERMOS_ESTRITAMENTE_ESPACIAIS = {"spacious", "echo"}

for _, row in df_eq_ouro.iterrows():
    texto = row['text']
    if texto in TERMOS_ESTRITAMENTE_ESPACIAIS:
        continue # Deixa a palavra livre para o Reverb capturar
        
    vocabulario_dsp[texto] = {
        'tipo':         'eq',
        'params':       row['param_values_list'],
        'consistencia': row['consistencia'],
        'eh_multipalavra': False
    }

for _, row in df_reverb_ouro.iterrows():
    texto = row['text']
    if texto not in vocabulario_dsp:
        vocabulario_dsp[texto] = {
            'tipo':           'reverb',
            'params':         row['param_values_list'],
            'consistencia':   row['consistencia'],
            'eh_multipalavra': ', ' in texto
        }

# Seleciona iteradores válidos para o contraste diametral InfoNCE.
palavras_aptas_hard_negative = {
    t for t in vocabulario_dsp
    if not vocabulario_dsp[t]['eh_multipalavra'] and t in PARES_OPOSTOS
}


# =============================================================================
# BLOCO 6 — SELEÇÃO DE NOTAS NSynth
# =============================================================================

print("\nCarregando notas base do NSynth Test...")
FAMILIAS_NSYNTH = ["bass", "brass", "flute", "guitar", "keyboard", "mallet", "organ", "reed", "string", "vocal"]

with open('nsynth-test/examples.json', 'r') as f:
    nsynth_meta = json.load(f)
print(f"  → {len(nsynth_meta)} notas no split de teste carregadas com sucesso.")

agrupamento_notas = {fam: [] for fam in FAMILIAS_NSYNTH}
for nota_id, info in nsynth_meta.items():
    familia = info.get('instrument_family_str')
    if familia in agrupamento_notas:
        agrupamento_notas[familia].append(nota_id)

notas_base_selecionadas = {}
total_notas_secas = 0
for fam, notas in agrupamento_notas.items():
    if notas:
        amostra = random.sample(notas, min(NOTAS_POR_FAMILIA, len(notas)))
        notas_base_selecionadas[fam] = amostra
        total_notas_secas += len(amostra)

print(f"  → Total de notas selecionadas: {total_notas_secas}")


# =============================================================================
# BLOCO 7 — GERAÇÃO DO BLUEPRINT (Matriz de Tarefas)
# =============================================================================
# JUSTIFICATIVA CIENTÍFICA: O pareamento aleatório dependente de lotes 
# estatísticos não assegura a ocorrência concorrente de âncora e oposição semântica. 
# A introdução de `id_par_hard_negative` indexa ambos os termos de forma determinística, 
# forçando o treinamento a concentrar a descida de gradiente inteiramente sobre as 
# diferenciações topológicas causadas pelo DSP.

print("\nGerando Blueprint...")
matriz_de_tarefas = []
contador_par = 0

for grupo_acustico, config in MAPEAMENTO_ESPARSO.items():
    familias_alvo = config["nsynth_families"]
    palavras_disponiveis = [p for p in config["adjetivos_alvo"] if p in vocabulario_dsp]

    for fam in familias_alvo:
        if fam not in notas_base_selecionadas:
            continue

        for nota_id in notas_base_selecionadas[fam]:
            candidatos_hn = [p for p in palavras_disponiveis if p in palavras_aptas_hard_negative]
            candidatos_simples = [p for p in palavras_disponiveis if p not in palavras_aptas_hard_negative]

            n_hn    = min(PARES_POR_NOTA, len(candidatos_hn))
            n_extra = PARES_POR_NOTA - n_hn

            ancoras_hn    = random.sample(candidatos_hn, n_hn) if n_hn > 0 else []
            ancoras_extra = random.sample(candidatos_simples, min(n_extra, len(candidatos_simples)))
            palavras_ancoras = ancoras_hn + ancoras_extra

            for palavra_ancora in palavras_ancoras:
                oposto  = PARES_OPOSTOS.get(palavra_ancora)
                info_a  = vocabulario_dsp[palavra_ancora]
                id_par  = f"par_{contador_par:05d}"
                contador_par += 1

                matriz_de_tarefas.append({
                    "nota_fonte":           nota_id,
                    "familia_instrumento":  fam,
                    "grupo_acustico":       grupo_acustico,
                    "descritor":            palavra_ancora,
                    "tipo_efeito":          info_a['tipo'],
                    "params_dsp":           info_a['params'],
                    "consistencia":         info_a['consistencia'],
                    "eh_negativo_dificil":  False,
                    "eh_multipalavra":      info_a['eh_multipalavra'],
                    "id_par_hard_negative": id_par,
                    "oposto_de":            None,
                })

                if oposto and oposto in vocabulario_dsp:
                    info_o = vocabulario_dsp[oposto]
                    matriz_de_tarefas.append({
                        "nota_fonte":           nota_id,
                        "familia_instrumento":  fam,
                        "grupo_acustico":       grupo_acustico,
                        "descritor":            oposto,
                        "tipo_efeito":          info_o['tipo'],
                        "params_dsp":           info_o['params'],
                        "consistencia":         info_o['consistencia'],
                        "eh_negativo_dificil":  True,
                        "eh_multipalavra":      info_o['eh_multipalavra'],
                        "id_par_hard_negative": id_par, 
                        "oposto_de":            palavra_ancora,
                    })

df_blueprint = pd.DataFrame(matriz_de_tarefas)


# =============================================================================
# BLOCO 8 — RELATÓRIO E EXPORTAÇÃO
# =============================================================================

total_estimado = len(df_blueprint) * 5

print("\n" + "=" * 70)
print("  BLUEPRINT FASE 1 — RELATÓRIO FINAL")
print("=" * 70)
print(f"\n  Notas fonte selecionadas:              {total_notas_secas:>8}")
print(f"  Entradas na matriz de tarefas:         {len(df_blueprint):>8}")
print(f"  └─ Âncoras (pares positivos):          {df_blueprint[~df_blueprint['eh_negativo_dificil']].shape[0]:>8}")
print(f"  └─ Hard Negatives estruturais:         {df_blueprint[df_blueprint['eh_negativo_dificil']].shape[0]:>8}")
print(f"  └─ Âncoras multi-palavra (reverb):     {df_blueprint[df_blueprint['eh_multipalavra']].shape[0]:>8}")
print(f"\n  Dataset estimado (×5 intensidades):    {total_estimado:>8} amostras")

print("\n  Distribuição por tipo de efeito DSP:")
print(df_blueprint.groupby(['tipo_efeito', 'eh_negativo_dificil']).size().rename("contagem").to_string())

print("\n  Distribuição por grupo acústico:")
print(df_blueprint.groupby('grupo_acustico').size().rename("contagem").to_string())

print("\n  Pares de Hard Negatives (amostra de validação):")
pares_demo = (
    df_blueprint[df_blueprint['id_par_hard_negative'].duplicated(keep=False)]
   .sort_values(['id_par_hard_negative', 'eh_negativo_dificil'])
   .head(8)[['nota_fonte', 'descritor', 'tipo_efeito', 'eh_negativo_dificil', 'id_par_hard_negative', 'consistencia']]
   .to_string(index=False)
)
print(pares_demo)

DIR_OUTPUT = "dataset_fase1_blueprints"
os.makedirs(DIR_OUTPUT, exist_ok=True)

path_blueprint = os.path.join(DIR_OUTPUT, "blueprint_fase1.parquet")
path_eq = os.path.join(DIR_OUTPUT, "vocabulario_eq_ouro.parquet")
path_reverb = os.path.join(DIR_OUTPUT, "vocabulario_reverb_ouro.parquet")

df_blueprint.to_parquet(path_blueprint, index=False)
df_eq_ouro.to_parquet(path_eq, index=False)
df_reverb_ouro.to_parquet(path_reverb, index=False)

print(f"\n  Arquivos exportados para a pasta '{DIR_OUTPUT}/':")
print("    -> blueprint_fase1.parquet")
print("    -> vocabulario_eq_ouro.parquet")
print("    -> vocabulario_reverb_ouro.parquet")
print("=" * 70)

Iniciando Fase 1 (Consolidada): Curadoria, Mapeamento Esparso e Hard Negatives

Baixando SocialFX do Hugging Face...
  EQ     ->  207 termos | range consistência: [0.501 - 0.931]
  Reverb ->  300 termos | range consensus: [5 - 11 palavras]

  Cobertura dos descritores-alvo:
    ✓ bright        [EQ     | consistência=0.873]
    ✓ clear         [EQ     | consistência=0.727]
    ✓ crisp         [EQ     | consistência=0.790]
    ✓ dark          [EQ     | consistência=0.880]
    ✗ distant       — ausente (verificar limiar ou split)
    ✗ dry           — ausente (verificar limiar ou split)
    ✓ echo          [REVERB | consensus=8]
    ✓ harsh         [EQ     | consistência=0.756]
    ✓ hollow        [EQ     | consistência=0.805]
    ✓ mellow        [EQ     | consistência=0.734]
    ✓ metallic      [EQ     | consistência=0.908]
    ✓ muffled       [EQ     | consistência=0.869]
    ✓ soft          [EQ     | consistência=0.927]
    ✓ spacious      [REVERB | consensus=7]
    ✓ tinny         [EQ

In [22]:
import os
import random
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm
from pedalboard import Pedalboard, PeakFilter, Reverb

print("Iniciando Fase 2: Motor DSP Unificado e Geração Matricial")
print("=" * 80)

# =============================================================================
# BLOCO 1 — CONFIGURAÇÕES DE DIRETÓRIO E ESCALA
# =============================================================================

# Diretórios de I/O
DIR_BLUEPRINTS = "dataset_fase1_blueprints"
DIR_AUDIO_IN = "nsynth-test/audio"
DIR_AUDIO_OUT = "dataset_fase2_final/audio"
DIR_META_OUT = "dataset_fase2_final"

os.makedirs(DIR_AUDIO_OUT, exist_ok=True)

# Escala de intensidades e seus respectivos modificadores textuais.
# JUSTIFICATIVA CIENTÍFICA: A injeção de advérbios correlacionados à magnitude
# do processamento DSP permite que o Joint Language Embedding (CLAP) aprenda 
# a topologia contínua da intensidade do efeito, não apenas a classe binária.
INTENSIDADES = [0.2, 0.4, 0.6, 0.8, 1.0]

MODIFICADORES = {
    0.2: ["slightly", "faintly", "a hint of"],
    0.4: ["mildly", "a bit", "marginally"],
    0.6: ["moderately", "somewhat", "partially"],
    0.8: ["mostly", "substantially", "largely"],
    1.0: ["", "", ""] # Intensidade máxima usa o adjetivo puro (ex: "a bright guitar")
}


# =============================================================================
# BLOCO 2 — MOTOR DSP UNIFICADO (Baseado na sua implementação original)
# =============================================================================

# Vetor de 40 frequências logarítmicas (padrão do SocialFX)
freqs_eq = np.logspace(np.log10(20), np.log10(20000), num=40)
LIMITE_CLIPPING = 0.95

def processar_dsp_monotonico(audio_data, sample_rate, tipo_efeito, parametros, intensidade):
    """
    Roteia o processamento baseado no tipo de efeito (EQ ou Reverb)
    aplicando escalonamento paramétrico linear, normalização de ganho e proteção Anti-Clipping.
    """
    
    if tipo_efeito == 'eq':
        filtros = []
        
        # 1. Encontra o pico máximo desenhado pelo humano (seja corte ou ganho)
        max_ganho = np.max(np.abs(parametros))
        
        # 2. Fator de normalização: Forçamos o pico do EQ para 15 dB (efeito dramático)
        fator_amplificacao = 15.0 / max_ganho if max_ganho > 0 else 1.0

        for i, freq in enumerate(freqs_eq):
            if i < len(parametros):
                # 3. Amplifica a curva e depois corta na proporção da intensidade (0.2 a 1.0)
                ganho_escalado = (parametros[i] * fator_amplificacao) * intensidade
                
                # Ignora filtros quase inaudíveis para poupar processamento (limiar ajustado para 0.5)
                if abs(ganho_escalado) > 0.5: 
                    filtros.append(PeakFilter(cutoff_frequency_hz=freq, gain_db=ganho_escalado, q=1.5))
        
        board = Pedalboard(filtros)
        
    elif tipo_efeito == 'reverb':
        # Reverb mais agressivo para garantir clusterização nas métricas
        room_size_calc = 0.2 + (intensidade * 0.8)  # De 0.2 a 1.0 (caudas enormes no máximo)
        wet_level_calc = intensidade * 0.9          # Deixa o som quase 100% "molhado" no máximo
        board = Pedalboard([
            Reverb(room_size=room_size_calc, wet_level=wet_level_calc, dry_level=1.0)
        ])
        
    else:
        raise ValueError(f"Tipo de efeito DSP desconhecido: {tipo_efeito}")

    # Conversão de segurança para Mono
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    # Processamento do sinal
    audio_processado = board(audio_data, sample_rate)

    # Proteção Digital Anti-Clipping (Normalização de Pico Condicional)
    pico_maximo = np.max(np.abs(audio_processado))
    if pico_maximo > LIMITE_CLIPPING:
        audio_processado = (audio_processado / pico_maximo) * LIMITE_CLIPPING

    return audio_processado


# =============================================================================
# BLOCO 3 — GERADOR DE PROMPTS SEMÂNTICOS
# =============================================================================

def gerar_prompt_semantico(palavra, familia_inst, intensidade, tipo_efeito):
    """
    Constrói a representação textual ancorada combinando o modificador de 
    magnitude, o descritor acústico e a fonte sonora.
    """
    modificador = random.choice(MODIFICADORES[intensidade])
    prefixo = f"{modificador} {palavra}".strip()
    contexto = familia_inst.replace('_', ' ') # ex: "synth lead"
    
    if tipo_efeito == 'eq':
        templates = [
            f"a {prefixo} {contexto}",
            f"{contexto} with a {prefixo} equalization",
            f"a recording of a {contexto} modified to sound {prefixo}"
        ]
    elif tipo_efeito == 'reverb':
        templates = [
            f"a {contexto} with {prefixo} reverberation",
            f"a {contexto} placed in a {prefixo} acoustic space",
            f"{contexto} processed to have a {prefixo} spatial effect"
        ]
    return random.choice(templates)


# =============================================================================
# BLOCO 4 — LAÇO DE PROCESSAMENTO E GERAÇÃO
# =============================================================================

print("Carregando Blueprint da Fase 1...")
try:
    df_blueprint = pd.read_parquet(os.path.join(DIR_BLUEPRINTS, "blueprint_fase1.parquet"))
except FileNotFoundError:
    raise RuntimeError("Blueprint não encontrado. Execute a Fase 1 primeiro.")

dataset_metadata = []
erros_leitura = 0

print(f"Iniciando renderização de {len(df_blueprint) * len(INTENSIDADES)} amostras de áudio...")

# Iteração linha a linha pelo Blueprint
for index, row in tqdm(df_blueprint.iterrows(), total=len(df_blueprint), desc="Processando DSP"):
    nota_id = row['nota_fonte']
    descritor = row['descritor']
    tipo_efeito = row['tipo_efeito']
    parametros = row['params_dsp']
    familia = row['familia_instrumento']
    
    audio_path = os.path.join(DIR_AUDIO_IN, f"{nota_id}.wav")
    
    try:
        audio_data, sr = sf.read(audio_path)
    except FileNotFoundError:
        erros_leitura += 1
        continue
        
    for i in INTENSIDADES:
        pct = int(i * 100)
        
        # 1. Aplica o processamento matemático
        audio_processado = processar_dsp_monotonico(
            audio_data=audio_data, 
            sample_rate=sr, 
            tipo_efeito=tipo_efeito, 
            parametros=parametros, 
            intensidade=i
        )
        
        # 2. Salva o arquivo em disco
        nome_arquivo = f"{tipo_efeito}_{descritor.replace(' ', '_')}_{pct}pct_{nota_id}.wav"
        caminho_salvo = os.path.join(DIR_AUDIO_OUT, nome_arquivo)
        sf.write(caminho_salvo, audio_processado, sr)
        
        # 3. Gera a legenda contrastiva (Prompt)
        prompt = gerar_prompt_semantico(descritor, familia, i, tipo_efeito)
        
        # 4. Registra no metadado final
        dataset_metadata.append({
            "file_name": nome_arquivo,
            "text": prompt,
            "keyword": descritor,
            "effect_type": tipo_efeito,
            "intensity": i,
            "original_note": nota_id,
            "is_hard_negative": row['eh_negativo_dificil'],
            "hard_negative_pair_id": row['id_par_hard_negative'],
            "consistency_score": row['consistencia']
        })

# Exportação do metadado final
df_treino_final = pd.DataFrame(dataset_metadata)
df_treino_final.to_csv(os.path.join(DIR_META_OUT, "metadata.csv"), index=False)

print("\n" + "=" * 80)
print("  RENDERIZAÇÃO CONCLUÍDA")
print("=" * 80)
print(f"  Total de arquivos de áudio gerados: {len(df_treino_final)}")
if erros_leitura > 0:
    print(f"  Aviso: {erros_leitura} notas do NSynth não foram encontradas no diretório fonte.")
print(f"  Metadados de treinamento salvos em: {os.path.join(DIR_META_OUT, 'metadata.csv')}")
print("  Dataset finalizado e pronto para ingestão no modelo CLAP.")

Iniciando Fase 2: Motor DSP Unificado e Geração Matricial
Carregando Blueprint da Fase 1...
Iniciando renderização de 12000 amostras de áudio...


Processando DSP: 100%|██████████████████████████████████████████████████████████████| 2400/2400 [13:24<00:00,  2.98it/s]


  RENDERIZAÇÃO CONCLUÍDA
  Total de arquivos de áudio gerados: 12000
  Metadados de treinamento salvos em: dataset_fase2_final/metadata.csv
  Dataset finalizado e pronto para ingestão no modelo CLAP.
